## Tarea 2.1: Tokenizador con Instrucciones

In [ ]:
import pandas as pd
import torch
import pickle
from collections import Counter

class Fechas2InstructTokenizer:
    """Tokenizador con tokens de instrucción para multitarea."""
    
    def __init__(self, train_es_file, train_en_file):
        # Leer archivos
        df_es = pd.read_csv(train_es_file)
        df_en = pd.read_csv(train_en_file)
        
        # Extraer vocabulario de ambos idiomas
        words = set()
        for text in list(df_es['txt']) + list(df_en['txt']):
            words.update(text.lower().split())
        
        words = sorted(list(words))
        
        # Crear diccionario
        self.word2index = {
            '<pad>': 0,
            '<sos>': 1,
            '<eos>': 2,
            # Tokens de instrucción
            '<transcribe_es>': 3,
            '<transcribe_en>': 4,
            '<translate_en_es>': 5,
            '<translate_es_en>': 6,
        }
        
        # Añadir palabras
        for i, word in enumerate(words, start=7):
            self.word2index[word] = i
        
        self.index2word = {v: k for k, v in self.word2index.items()}
        self.vocab_size = len(self.word2index)
        
        print(f"Vocabulario con instrucciones: {self.vocab_size} tokens")
        print(f"Tokens de instrucción:")
        for action in ['<transcribe_es>', '<transcribe_en>', '<translate_en_es>', '<translate_es_en>']:
            print(f"  {action}: {self.word2index[action]}")
    
    def encode(self, text, action=None, seq_len=-1):
        """Codifica texto con token de acción opcional.
        
        Args:
            text: Texto a codificar
            action: Token de acción (transcribe_es, transcribe_en, etc.)
            seq_len: Longitud máxima
        """
        tokens = [self.word2index['<sos>']]
        
        # Añadir token de acción si se proporciona
        if action is not None:
            action_token = f'<{action}>'
            if action_token in self.word2index:
                tokens.append(self.word2index[action_token])
        
        # Añadir palabras
        for word in text.lower().split():
            if word in self.word2index:
                tokens.append(self.word2index[word])
        
        tokens.append(self.word2index['<eos>'])
        
        # Padding
        if seq_len > len(tokens):
            tokens = tokens + [self.word2index['<pad>']] * (seq_len - len(tokens))
        
        return torch.tensor(tokens)
    
    def decode(self, indices):
        """Decodifica secuencia de índices."""
        if isinstance(indices, torch.Tensor):
            indices = indices.tolist()
        
        words = []
        for idx in indices:
            if idx in self.index2word:
                word = self.index2word[idx]
                # Excluir tokens especiales (pero mantener tokens de instrucción para debug)
                if word not in ['<pad>', '<sos>', '<eos>']:
                    words.append(word)
        
        return ' '.join(words)

# Crear tokenizador
tokenizer_instruct = Fechas2InstructTokenizer(
    '../fechas2/fechas2_train.es.csv',
    '../fechas2/fechas2_train.en.csv'
)

# Guardar
with open('fechas2_tokenizer_instruct.pkl', 'wb') as f:
    pickle.dump(tokenizer_instruct, f)

print("\nTokenizador guardado en 'fechas2_tokenizer_instruct.pkl'")

### Pruebas del Tokenizador

In [ ]:
# Probar tokenizador con diferentes acciones
ejemplos = [
    ("por favor el siguiente jueves", "transcribe_es"),
    ("please next thursday", "transcribe_en"),
    ("please next thursday", "translate_en_es"),
    ("por favor el siguiente jueves", "translate_es_en"),
]

print("Pruebas del tokenizador con instrucciones:\n")
for text, action in ejemplos:
    encoded = tokenizer_instruct.encode(text, action=action)
    decoded = tokenizer_instruct.decode(encoded)
    print(f"Texto: {text}")
    print(f"Acción: {action}")
    print(f"Codificado: {encoded.tolist()[:10]}...")
    print(f"Decodificado: {decoded}")
    print()

## Tarea 2.2: Dataset Multitarea

In [ ]:
import torchaudio
import os
import numpy as np

class Fechas2MultiTaskDataset(torch.utils.data.Dataset):
    """Dataset multitarea con 4 acciones posibles."""
    
    def __init__(self, train_es_csv, train_en_csv, tokenizer, 
                 audio_len=4*16000, transforms=None, max_text_len=25):
        self.tokenizer = tokenizer
        self.audio_len = audio_len
        self.transforms = transforms if transforms is not None else []
        self.max_text_len = max_text_len
        
        # Leer ambos archivos
        df_es = pd.read_csv(train_es_csv)
        df_en = pd.read_csv(train_en_csv)
        
        # Crear ejemplos para todas las combinaciones de tareas
        self.examples = []
        
        # Para cada archivo en español
        for idx, row in df_es.iterrows():
            # Transcribe ES: audio_es -> text_es
            self.examples.append({
                'audio': row['wav'],
                'text': row['txt'],
                'action': 'transcribe_es',
                'lang': 'es'
            })
            
            # Translate ES->EN: audio_es -> text_en (si existe correspondiente)
            if idx < len(df_en):
                self.examples.append({
                    'audio': row['wav'],
                    'text': df_en.iloc[idx]['txt'],  # Texto en inglés
                    'action': 'translate_es_en',
                    'lang': 'es'
                })
        
        # Para cada archivo en inglés
        for idx, row in df_en.iterrows():
            # Transcribe EN: audio_en -> text_en
            self.examples.append({
                'audio': row['wav'],
                'text': row['txt'],
                'action': 'transcribe_en',
                'lang': 'en'
            })
            
            # Translate EN->ES: audio_en -> text_es (si existe correspondiente)
            if idx < len(df_es):
                self.examples.append({
                    'audio': row['wav'],
                    'text': df_es.iloc[idx]['txt'],  # Texto en español
                    'action': 'translate_en_es',
                    'lang': 'en'
                })
        
        # Directorio base
        self.base_dir = os.path.dirname(train_es_csv)
        
        print(f"Dataset multitarea creado: {len(self.examples)} ejemplos")
        
        # Estadísticas
        actions = [ex['action'] for ex in self.examples]
        print("\nDistribución de acciones:")
        for action in ['transcribe_es', 'transcribe_en', 'translate_en_es', 'translate_es_en']:
            count = actions.count(action)
            print(f"  {action}: {count} ({count/len(actions)*100:.1f}%)")
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        ex = self.examples[idx]
        
        # Cargar audio
        audio_path = os.path.join(self.base_dir, ex['audio'])
        if not os.path.exists(audio_path):
            audio_path = os.path.join('..', ex['audio'])
        
        x, fs = torchaudio.load(audio_path)
        
        if x.shape[1] < self.audio_len:
            x = torch.nn.functional.pad(x, (0, self.audio_len-x.shape[1]), value=0)
        else:
            x = x[:, :self.audio_len]
        
        x = x[0].numpy()
        
        # Aplicar augmentation
        for t in self.transforms:
            x = t(x)
        
        # Tokenizar con acción
        y = self.tokenizer.encode(ex['text'], action=ex['action'], seq_len=self.max_text_len)
        
        return torch.tensor(x, dtype=torch.float32), y

# Crear dataset (reutilizando las clases de augmentation de la tarea 1)
trainset_multitask = Fechas2MultiTaskDataset(
    '../fechas2/fechas2_train.es.csv',
    '../fechas2/fechas2_train.en.csv',
    tokenizer_instruct,
    transforms=[NoiseAug(prob=0.3), RIRAug(prob=0.3)]
)

## Tarea 2.3: Entrenamiento y Evaluación

In [ ]:
# Importar modelo de tarea 1.3 (AudioTransformer)
# Crear modelo con nuevo vocabulario

model_config_multitask = {
    'vocab_size': tokenizer_instruct.vocab_size,
    'd_model': 256,
    'nb_layers': 6,
    'd_ff': 512,
    'n_heads': 8,
    'd_head': 32,
    'dropout': 0.1,
    'seq_len': 500,
    'feat_dim': 80
}

# NOTA: Usar la clase AudioTransformer definida en tarea 1.3
# pero ajustada para usar el nuevo tokenizer

model_multitask = AudioTransformer(**model_config_multitask)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_multitask.to(device)

opt = torch.optim.Adam(model_multitask.parameters(), lr=3e-4)

nb_epochs = 10
batch_size = 16

trainloader_multitask = torch.utils.data.DataLoader(
    trainset_multitask,
    batch_size=batch_size,
    shuffle=True
)

print(f"Iniciando entrenamiento multitarea...")

# Entrenamiento
model_multitask.train()
losses_multitask = []

for e in range(nb_epochs):
    epoch_loss = 0
    for batch_idx, (x, y) in enumerate(trainloader_multitask):
        x = x.to(device)
        y = y.to(device)
        
        opt.zero_grad()
        loss = model_multitask.loss(x, y)
        loss.backward()
        opt.step()
        
        epoch_loss += loss.item()
        
        if (batch_idx + 1) % 200 == 0:
            print(f'  Batch {batch_idx+1}/{len(trainloader_multitask)}: loss={loss.item():.4f}')
    
    avg_loss = epoch_loss / len(trainloader_multitask)
    losses_multitask.append(avg_loss)
    print(f'Epoch {e+1}/{nb_epochs}: avg_loss={avg_loss:.4f}')

torch.save({'model': model_multitask.state_dict(), 'opt': opt.state_dict(), 
            'config': model_config_multitask}, 'model_fechas2_multitask.pt')
print("Modelo multitarea guardado en 'model_fechas2_multitask.pt'")

### Evaluar con fechas2_test_instruct.csv

In [ ]:
import jiwer

# Dataset de test con instrucciones
class Fechas2InstructTestDataset(torch.utils.data.Dataset):
    def __init__(self, csv_file, tokenizer, audio_len=4*16000, max_text_len=25):
        self.df = pd.read_csv(csv_file)
        self.tokenizer = tokenizer
        self.audio_len = audio_len
        self.max_text_len = max_text_len
        self.csv_dir = os.path.dirname(csv_file)
        print(f"Test instruct dataset: {len(self.df)} ejemplos")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        audio_path = os.path.join(self.csv_dir, row['wav'])
        if not os.path.exists(audio_path):
            audio_path = os.path.join('..', row['wav'])
        
        x, fs = torchaudio.load(audio_path)
        
        if x.shape[1] < self.audio_len:
            x = torch.nn.functional.pad(x, (0, self.audio_len-x.shape[1]), value=0)
        else:
            x = x[:, :self.audio_len]
        
        return x[0], row['action'], row['txt']

testset_instruct = Fechas2InstructTestDataset(
    '../fechas2/fechas2_test_instruct.csv',
    tokenizer_instruct
)

# Evaluar
model_multitask.eval()
hyp_multitask = []
ref_multitask = []

print("Evaluando modelo multitarea...")

for i, (x, action, text_ref) in enumerate(testset_instruct):
    x = x.to(device)
    
    # Generar con token de acción
    action_token = tokenizer_instruct.word2index[f'<{action}>']
    
    # Generar predicción (versión modificada que empieza con acción)
    y_start = [tokenizer_instruct.word2index['<sos>'], action_token]
    
    with torch.no_grad():
        enc = model_multitask.encoder(x[None,...].to(device))
        
        while y_start[-1] != tokenizer_instruct.word2index['<eos>'] and len(y_start) < 30:
            logits = model_multitask.decoder(torch.tensor(y_start).unsqueeze(0).to(device), enc)
            y_start.append(logits.argmax(-1)[:,-1].item())
    
    hyp = tokenizer_instruct.decode(y_start)
    # Remover token de acción del resultado
    hyp = hyp.replace(f'<{action}>', '').strip()
    
    hyp_multitask.append(hyp)
    ref_multitask.append(text_ref)
    
    if i < 20:
        print(f"\nEjemplo {i+1} ({action}):")
        print(f"  REF: {text_ref}")
        print(f"  HYP: {hyp}")

# Calcular WER
out_multitask = jiwer.process_words(ref_multitask, hyp_multitask)
print(f"\n{'='*50}")
print(f"WER Multitarea: {out_multitask.wer:.2%}")
print(f"Substituciones: {out_multitask.substitutions}")
print(f"Deleciones: {out_multitask.deletions}")
print(f"Inserciones: {out_multitask.insertions}")
print(f"{'='*50}")

# Guardar resultados
results_multitask = pd.DataFrame({
    'action': [row['action'] for _, row in testset_instruct.df.iterrows()],
    'reference': ref_multitask,
    'hypothesis': hyp_multitask
})
results_multitask.to_csv('results_multitask.csv', index=False)
print("\nResultados guardados en 'results_multitask.csv'")

### Análisis por tipo de tarea

In [ ]:
# Analizar WER por tipo de acción
import matplotlib.pyplot as plt

results_df = pd.read_csv('results_multitask.csv')

print("WER por tipo de acción:\n")
for action in ['transcribe_es', 'transcribe_en', 'translate_en_es', 'translate_es_en']:
    mask = results_df['action'] == action
    refs = results_df[mask]['reference'].tolist()
    hyps = results_df[mask]['hypothesis'].tolist()
    
    if len(refs) > 0:
        out = jiwer.process_words(refs, hyps)
        print(f"  {action}: WER = {out.wer:.2%} ({len(refs)} ejemplos)")